# Actividad: XGBoost para clasificación — del boosting manual a la predicción de abandono de clientes

En esta actividad se utiliza XGBoost para resolver un problema de clasificación binaria, que introduce preguntas que no aparecían en regresión:

- ¿Cómo puede aplicarse el boosting a un modelo cuya salida es una probabilidad?
- ¿Qué estrategia conviene cuando una de las clases es mucho menos frecuente que la otra?
- ¿Es el *accuracy* una métrica adecuada? ¿Y el umbral de 0.5 para asignar la clase?

La actividad se divide en dos partes:

1. Construcción de la intuición de forma manual, con un conjunto de datos sintético en dos dimensiones, antes de utilizar una librería especializada.
2. Aplicación de `XGBClassifier` a un conjunto de datos real de *churn* (abandono de clientes), analizando los hiperparámetros más relevantes y las métricas apropiadas para un problema desbalanceado.

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
# Si esta celda falla con "No module named 'xgboost'", corran antes, en una
# celda o terminal: pip install xgboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import make_moons
from sklearn.metrics import (accuracy_score, roc_auc_score, f1_score, log_loss,
                             precision_score, recall_score, confusion_matrix,
                             classification_report, precision_recall_curve, roc_curve)
from xgboost import XGBClassifier

print("XGBoost cargado correctamente.")

## Parte 0 — El boosting aplicado a clasificación

En regresión, cada ronda de boosting consiste en: *predicción actual → residual (`y − predicción`) → árbol que ajusta ese residual → suma con un factor de aprendizaje reducido*. En clasificación el procedimiento es equivalente, con una diferencia: el modelo no acumula una predicción directa de `y`, sino una predicción en **log-odds** (el logaritmo del cociente de probabilidades). La conversión de log-odds a probabilidad se realiza con la función **sigmoide** $\sigma(z) = 1 / (1 + e^{-z})$.

1. Se parte de una predicción constante en log-odds, $\log\frac{p_0}{1-p_0}$, donde $p_0$ es la proporción de la clase positiva. En términos de probabilidad, equivale a asignar la tasa base a todas las observaciones.
2. Para cada observación se calcula el **pseudo-residual** `y − σ(predicción)`: la diferencia entre la etiqueta real (0 o 1) y la probabilidad que el modelo le asigna en ese momento. Corresponde al gradiente de la *log-loss*, de donde proviene el nombre *gradient boosting*.
3. Se entrena un árbol de **regresión de poca profundidad** para ajustar esos pseudo-residuales.
4. Se suma su predicción, multiplicada por el `learning_rate`, a la predicción acumulada **en log-odds**.
5. Se repite el proceso; los pseudo-residuales disminuyen en cada iteración.

Al finalizar, la predicción acumulada en log-odds se transforma con la sigmoide para obtener una probabilidad, que se compara con el umbral de 0.5 (valor por defecto) para asignar la clase. A continuación se implementa este procedimiento de forma manual sobre un conjunto de datos sintético en dos dimensiones.

In [ ]:
# ============================================================
# Ejemplo sintético — no modificar
# ============================================================
X_toy, y_toy = make_moons(n_samples=300, noise=0.30, random_state=11)

def sigmoide(z):
    return 1.0 / (1.0 + np.exp(-z))

learning_rate = 0.3
n_rondas = 30

p0 = y_toy.mean()
log_odds_inicial = np.log(p0 / (1 - p0))
pred = np.full(len(y_toy), log_odds_inicial)          # predicción acumulada en log-odds

# malla para representar la frontera de decisión
xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 300), np.linspace(-1.2, 1.8, 300))
malla = np.c_[xx.ravel(), yy.ravel()]
pred_malla = np.full(len(malla), log_odds_inicial)

log_loss_por_ronda = [log_loss(y_toy, sigmoide(pred), labels=[0, 1])]
snapshots = {}

for ronda in range(1, n_rondas + 1):
    prob = sigmoide(pred)                              # paso 1: log-odds -> probabilidad
    residual = y_toy - prob                            # paso 2: pseudo-residual (gradiente de la log-loss)
    arbolito = DecisionTreeRegressor(max_depth=2, random_state=0)
    arbolito.fit(X_toy, residual)                      # paso 3: árbol de regresión sobre el pseudo-residual
    pred = pred + learning_rate * arbolito.predict(X_toy)          # paso 4: suma en log-odds con factor reducido
    pred_malla = pred_malla + learning_rate * arbolito.predict(malla)
    log_loss_por_ronda.append(log_loss(y_toy, sigmoide(pred), labels=[0, 1]))
    if ronda in (1, 3, 10, 30):
        snapshots[ronda] = sigmoide(pred_malla).reshape(xx.shape).copy()

# --- Figura 1: evolución de la frontera de decisión ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, (ronda, prob_grid) in zip(axes, snapshots.items()):
    ax.contourf(xx, yy, prob_grid, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.75)
    ax.contour(xx, yy, prob_grid, levels=[0.5], colors="black", linewidths=1.5)
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_toy, cmap="RdBu_r", edgecolor="k", s=18, linewidth=0.4)
    ax.set_title(f"tras {ronda} ronda(s)")
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Evolución de la frontera de decisión por ronda (línea negra: probabilidad 0.5)")
plt.tight_layout()
plt.show()

# --- Figura 2: log-loss por ronda ---
plt.figure(figsize=(8, 4.5))
plt.plot(range(len(log_loss_por_ronda)), log_loss_por_ronda, "o-", color="darkred")
plt.xlabel("ronda de boosting"); plt.ylabel("log-loss sobre los datos")
plt.title("Reducción de la log-loss en cada ronda de boosting")
plt.grid(alpha=0.3)
plt.show()

acc_toy = accuracy_score(y_toy, (sigmoide(pred) > 0.5).astype(int))
print(f"log-loss -- ronda 1: {log_loss_por_ronda[1]:.3f}   ronda 30: {log_loss_por_ronda[-1]:.3f}")
print(f"accuracy tras 30 rondas: {acc_toy:.3f}")

**Observación.** Cada `arbolito` tiene profundidad 2 y, de forma aislada, no separa las clases. Sin embargo, al sumar 30 de ellos —cada uno corrigiendo el error de probabilidad de los anteriores— la frontera de decisión pasa de una recta a una curva que sigue la forma de las dos lunas. Nótese que la frontera presenta un aspecto escalonado: es la suma de árboles muy simples, no una función suave.

Este es el procedimiento que `XGBClassifier` ejecuta internamente: acumula un gran número de árboles de regresión débiles sobre los pseudo-residuales, en el espacio de log-odds, y aplica la sigmoide al final.

## Parte 1 — El conjunto de datos: abandono de clientes (*churn*)

Se utiliza el conjunto **Telco Customer Churn**, habitual para practicar clasificación con variables mixtas. Contiene aproximadamente 7,000 clientes de una empresa de telecomunicaciones, y el objetivo es determinar si el cliente se dio de baja (`Churn`). Algunas de sus columnas:

| Columna | Significado |
|---|---|
| `tenure` | meses que lleva como cliente |
| `Contract` | tipo de contrato (mes a mes / 1 año / 2 años) |
| `MonthlyCharges` | cargo mensual |
| `TotalCharges` | cargo total acumulado |
| `InternetService` | tipo de internet (DSL / fibra / ninguno) |
| `TechSupport`, `OnlineSecurity`, ... | servicios adicionales contratados |
| `PaymentMethod` | forma de pago |
| `Churn` | **si el cliente se dio de baja (lo que queremos predecir)** |

El conjunto permite contrastar el modelo con el conocimiento del dominio: un cliente con contrato mensual, poca antigüedad y cargos elevados presenta, a priori, mayor propensión al abandono. Solo el **~27 % de los clientes se da de baja**, por lo que las clases están desbalanceadas.

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# TotalCharges viene como texto y trae 11 celdas en blanco (clientes con tenure=0)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)
df = df.drop(columns=["customerID"])          # identificador único, sin valor predictivo

print(df.shape)
print(df["Churn"].value_counts(normalize=True).round(3).to_dict())
df.head()

In [ ]:
# ============================================================
# PREPARACIÓN — no modifiquen esta celda
# ============================================================
df_cod = df.copy()
df_cod["Churn"] = (df_cod["Churn"] == "Yes").astype(int)      # objetivo -> 0 / 1

# el resto de columnas de texto son categóricas -> columnas 0/1
cat_cols = df_cod.select_dtypes("object").columns.tolist()
df_cod = pd.get_dummies(df_cod, columns=cat_cols, drop_first=True)

X = df_cod.drop(columns=["Churn"])
y = df_cod["Churn"]

# stratify=y mantiene la misma proporción de churn en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Entrenamiento:", X_train.shape, "  Prueba:", X_test.shape)
print(f"Proporción de churn -- train: {y_train.mean():.3f}   test: {y_test.mean():.3f}")

## Parte 2 — Modelos de referencia: regresión logística y árbol sin restricciones

Antes de aplicar XGBoost, establezcan los valores de referencia con los que se comparará. Se emplearán **dos** métricas desde el inicio:

- **accuracy**: proporción de aciertos. Es de lectura directa, pero **poco informativa con clases desbalanceadas**: un modelo que prediga siempre la clase mayoritaria alcanza ya un ~73 %.
- **ROC-AUC**: mide la capacidad del modelo para *ordenar* las observaciones según su riesgo (probabilidad de que un cliente que efectivamente se dio de baja reciba una puntuación mayor que uno que no). 0.5 corresponde al azar y 1.0 a la separación perfecta. No depende del umbral ni de la proporción de clases.

In [ ]:
# modelo trivial: predecir siempre la clase mayoritaria (ningún cliente se da de baja)
acc_trivial = accuracy_score(y_test, np.zeros_like(y_test))
print(f"Predicción constante    -- accuracy: {acc_trivial:.4f}   ROC-AUC: 0.500 (no discrimina)")

# regresión logística (con estandarización previa para asegurar la convergencia)
log_reg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
log_reg.fit(X_train, y_train)
p_log = log_reg.predict_proba(X_test)[:, 1]
print(f"Regresión logística     -- accuracy: {accuracy_score(y_test, log_reg.predict(X_test)):.4f}   "
      f"ROC-AUC: {roc_auc_score(y_test, p_log):.4f}")

# árbol de decisión sin restricciones
arbol_default = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
p_arbol = arbol_default.predict_proba(X_test)[:, 1]
print(f"Árbol sin restricciones -- accuracy entrenamiento: {accuracy_score(y_train, arbol_default.predict(X_train)):.4f}  "
      f"(profundidad {arbol_default.get_depth()}, {arbol_default.get_n_leaves()} hojas)")
print(f"Árbol sin restricciones -- accuracy prueba: {accuracy_score(y_test, arbol_default.predict(X_test)):.4f}   "
      f"ROC-AUC: {roc_auc_score(y_test, p_arbol):.4f}")

**Observen** que el árbol sin restricciones ajusta casi por completo el conjunto de entrenamiento, pero en prueba su ROC-AUC queda muy por debajo de la regresión logística, y su accuracy apenas supera —o no alcanza— a la del modelo trivial. Es el patrón de sobreajuste ya conocido. La regresión logística, en cambio, constituye un modelo de referencia **sólido** en este problema: una parte considerable de la señal es aproximadamente lineal.

## Parte 3 — XGBoost con hiperparámetros por defecto

**Ejercicio.** Creen un `XGBClassifier(random_state=42, eval_metric="logloss")`, entrénenlo con `X_train, y_train` y calculen su **accuracy** y **ROC-AUC** sobre el conjunto de prueba. Para el ROC-AUC se necesitan las probabilidades: `modelo.predict_proba(X_test)[:, 1]`.

In [ ]:
# TODO: creen y entrenen un XGBClassifier(random_state=42, eval_metric="logloss")
# llamado xgb_default, y calculen su accuracy y ROC-AUC de prueba.

# xgb_default = ...
# xgb_default.fit(...)
# p_xgb_default = xgb_default.predict_proba(X_test)[:, 1]
# acc_xgb_default = ...
# auc_xgb_default = ...
# print(...)

**Para registrar:** ¿superó XGBoost a la regresión logística en ROC-AUC? Es posible que **no**. Con los hiperparámetros por defecto, XGBoost tiende a sobreajustar en un conjunto como este, con señal predominantemente lineal y un número reducido de variables. A diferencia del caso de regresión, aquí los valores por defecto no son suficientes: el modelo requiere ajuste, que es el objeto de las siguientes secciones.

## Parte 4 — El proceso de aprendizaje: curva de error por iteración

Como en el ejemplo manual, XGBoost incorpora los árboles de uno en uno. Es posible **registrar la log-loss en cada iteración**, tanto en entrenamiento como en un conjunto de evaluación, mediante `eval_set`. Esta celda utiliza deliberadamente un `learning_rate` alto y árboles de profundidad media para que el sobreajuste resulte visible.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
modelo_curva = XGBClassifier(
    n_estimators=400, learning_rate=0.3, max_depth=6,
    random_state=42, eval_metric="logloss"
)
modelo_curva.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False
)
resultados_curva = modelo_curva.evals_result()
loss_train = resultados_curva["validation_0"]["logloss"]
loss_test = resultados_curva["validation_1"]["logloss"]

plt.figure(figsize=(8, 5))
plt.plot(loss_train, label="entrenamiento")
plt.plot(loss_test, label="prueba")
mejor_ronda = int(np.argmin(loss_test))
plt.axvline(mejor_ronda, color="gray", linestyle="--", linewidth=1,
            label=f"mejor ronda en prueba ({mejor_ronda})")
plt.xlabel("ronda de boosting (árbol #)")
plt.ylabel("log-loss")
plt.title("learning_rate=0.3, max_depth=6 -- boosting sin regularización")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Mejor log-loss de prueba: {min(loss_test):.4f}, en la ronda {mejor_ronda}")
print(f"log-loss de prueba en la última ronda ({len(loss_test)}): {loss_test[-1]:.4f}")

**Interpretación.** El error de entrenamiento decrece a lo largo de toda la curva (un mayor número de árboles siempre ajusta mejor el entrenamiento). El error de **prueba**, en cambio, alcanza su mínimo pronto y después aumenta: a partir de ese punto, cada árbol adicional solo contribuye a memorizar ruido. Es el mismo fenómeno de sobreajuste, ahora expresado por iteración.

## Parte 5 — Hiperparámetros de control: `learning_rate`, `max_depth` y criterio de parada

Los mismos tres mecanismos que en regresión:

- **`learning_rate` reducido** (p. ej. 0.03): cada árbol aporta una corrección menor, lo que exige más iteraciones pero reduce el riesgo de sobreajuste.
- **`max_depth` reducido** (2–4): en boosting los árboles deben ser deliberadamente simples (*weak learners*).
- **`early_stopping_rounds`**: en lugar de fijar `n_estimators` de antemano, se indica a XGBoost que detenga el entrenamiento si la métrica de validación no mejora durante N iteraciones consecutivas.

**Ejercicio.** Separen un conjunto de *validación* a partir de `X_train`; no debe usarse `X_test` para determinar el punto de parada:

```python
X_train2, X_val, y_train2, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
```

Entrenen un `XGBClassifier` con `n_estimators=2000`, `learning_rate=0.03`, `max_depth=3`, `eval_metric="auc"` y `early_stopping_rounds=50`, usando `eval_set=[(X_val, y_val)]`. Evalúen el modelo final sobre el `X_test` original (accuracy y ROC-AUC).

In [ ]:
# TODO: separen un set de validación de X_train, entrenen con early stopping
# (métrica auc), y evalúen el modelo final (llámenlo xgb_early) contra X_test.

# X_train2, X_val, y_train2, y_val = train_test_split(
#     X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
# xgb_early = XGBClassifier(n_estimators=2000, learning_rate=0.03, max_depth=3,
#                           random_state=42, eval_metric="auc", early_stopping_rounds=50)
# xgb_early.fit(X_train2, y_train2, eval_set=[(X_val, y_val)], verbose=False)
# print("Mejor ronda encontrada:", xgb_early.best_iteration)
# ...

## Parte 6 — Clases desbalanceadas: `scale_pos_weight` y elección de métrica

Solo el ~27 % de los clientes se da de baja. Observen la **matriz de confusión** del modelo con parada temprana (celda siguiente): es previsible que clasifique correctamente casi todos los casos de "no baja", pero que no identifique a una proporción considerable de los clientes que sí se dan de baja (**falsos negativos**). Desde el punto de vista del negocio, esos son los casos de mayor costo, ya que corresponden a clientes potencialmente retenibles.

Dos herramientas:

- **`scale_pos_weight`**: indica a XGBoost el peso relativo de cada observación de la clase positiva frente a una de la clase negativa. El valor neutro es 1; el valor que equilibra las clases es `n_negativos / n_positivos` (~2.7 en este caso). Incrementarlo hace que el modelo prediga la clase positiva con mayor frecuencia: **aumenta el *recall* y reduce la *precision***, y por lo general disminuye la accuracy.
- **Considerar precision, recall y F1**, no solo accuracy. El *recall* indica qué proporción de los clientes que se dieron de baja fue detectada; la *precision*, qué proporción de los clientes señalados como "baja" se dio de baja efectivamente.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
spw_balanceado = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight que equilibra las clases: {spw_balanceado:.2f}\n")

valores_spw = [1, 2, spw_balanceado, 2 * spw_balanceado]
filas = []
for spw in valores_spw:
    m = XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=3,
                      random_state=42, eval_metric="logloss", scale_pos_weight=spw)
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    proba = m.predict_proba(X_test)[:, 1]
    filas.append({
        "scale_pos_weight": round(spw, 2),
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })
print(pd.DataFrame(filas).round(4).to_string(index=False))

# matrices de confusión: neutro (spw=1) vs equilibrado
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, spw in zip(axes, [1, spw_balanceado]):
    m = XGBClassifier(n_estimators=400, learning_rate=0.03, max_depth=3,
                      random_state=42, eval_metric="logloss", scale_pos_weight=spw)
    m.fit(X_train, y_train)
    cm = confusion_matrix(y_test, m.predict(X_test))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center", fontsize=13)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["pred: no", "pred: churn"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["real: no", "real: churn"])
    ax.set_title(f"scale_pos_weight = {spw:.2f}")
plt.tight_layout()
plt.show()

**Interpretación.** Al aumentar `scale_pos_weight`, la celda de **falsos negativos** (real: baja / predicho: no) se reduce —se detectan más clientes en riesgo— a costa de un mayor número de **falsos positivos** y una menor accuracy. El **ROC-AUC apenas varía**: la capacidad del modelo para ordenar a los clientes por riesgo es la misma; lo que cambia es el punto de corte. El valor apropiado depende del costo relativo de cada tipo de error para el negocio.

## Parte 7 — El umbral de decisión

`modelo.predict(...)` aplica el umbral de 0.5: si la probabilidad estimada de baja es ≥ 0.5, se predice "baja". Ese valor es una convención, no un requisito. Si el costo de una acción de retención es bajo y el de perder al cliente es alto, puede convenir señalar como "en riesgo" a toda observación con probabilidad superior a, por ejemplo, 0.3.

La **curva precision-recall** representa ese compromiso para todos los umbrales de forma simultánea. La celda siguiente la genera para `xgb_early` y marca el umbral que maximiza el F1.

**Ejercicio.** A partir de las probabilidades `p_xgb_early`, seleccionen un umbral distinto de 0.5 (por ejemplo, el que maximiza el F1, o uno que garantice un recall ≥ 0.7), construyan las predicciones `(p_xgb_early >= umbral)` y comparen la **matriz de confusión** y el `classification_report` con los correspondientes al umbral de 0.5.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
prec, rec, umbrales = precision_recall_curve(y_test, p_xgb_early)
f1_por_umbral = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-12)
umbral_mejor_f1 = umbrales[np.argmax(f1_por_umbral)]

plt.figure(figsize=(7, 5))
plt.plot(rec, prec, lw=2)
i = np.argmax(f1_por_umbral)
plt.scatter(rec[i], prec[i], color="red", zorder=5,
            label=f"mejor F1 @ umbral {umbral_mejor_f1:.2f}")
plt.xlabel("recall (de los que se fueron, ¿cuántos detectamos?)")
plt.ylabel("precision (de los que marcamos, ¿cuántos se fueron?)")
plt.title("Curva precision-recall -- xgb_early")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(f"Umbral que maximiza F1: {umbral_mejor_f1:.3f}")

In [ ]:
# TODO: comparen el umbral 0.5 contra un umbral que ustedes elijan.

# pred_05    = (p_xgb_early >= 0.5).astype(int)
# pred_nuevo = (p_xgb_early >= umbral_mejor_f1).astype(int)
# print("--- umbral 0.5 ---")
# print(confusion_matrix(y_test, pred_05))
# print(classification_report(y_test, pred_05, digits=3))
# print("--- umbral nuevo ---")
# ...

## Parte 8 — Búsqueda de hiperparámetros con `RandomizedSearchCV`

Hasta aquí se han analizado los hiperparámetros de forma individual. En la práctica se exploran varias combinaciones de forma conjunta mediante validación cruzada. Dado que evaluar todas las combinaciones (`GridSearchCV`) tiene un costo computacional elevado, se emplea `RandomizedSearchCV`, que evalúa una muestra aleatoria del espacio de búsqueda. En un problema con clases desbalanceadas es importante fijar **`scoring="roc_auc"`** en lugar de `"accuracy"`.

**Ejercicio.** Con el `param_dist` definido a continuación, construyan un `RandomizedSearchCV` sobre `XGBClassifier(random_state=42, eval_metric="logloss")` con `n_iter=40`, `cv=5`, `scoring="roc_auc"`, `random_state=42` y `n_jobs=-1`. Ajústenlo sobre `X_train, y_train` y guarden `busqueda.best_estimator_` en `xgb_tuned`.

In [ ]:
# ============================================================
# Ya lista — no modificar
# ============================================================
param_dist = {
    "n_estimators": [200, 300, 500, 800],
    "max_depth": [2, 3, 4, 5],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0],
    "reg_lambda": [0, 1, 5, 20, 50],
    "reg_alpha": [0, 1, 5, 20],
    "scale_pos_weight": [1, 2, 2.7],
}

In [ ]:
# TODO: construyan y ajusten el RandomizedSearchCV (llámenlo busqueda),
# y guarden el mejor modelo en xgb_tuned.

# busqueda = RandomizedSearchCV(
#     XGBClassifier(random_state=42, eval_metric="logloss"),
#     param_distributions=param_dist, n_iter=40, cv=5,
#     scoring="roc_auc", random_state=42, n_jobs=-1)
# busqueda.fit(X_train, y_train)
# print("Mejores parámetros:", busqueda.best_params_)
# xgb_tuned = busqueda.best_estimator_

## Parte 9 — Importancia de las variables

Como todo modelo basado en árboles, XGBoost puede informar en qué medida se utilizó cada variable en las divisiones. Constituye una forma rápida de interpretación, aunque el modelo sea la suma de cientos de árboles.

In [ ]:
importancias = pd.Series(xgb_tuned.feature_importances_, index=X.columns)
importancias = importancias.sort_values().tail(15)   # las 15 más usadas

plt.figure(figsize=(7, 6))
importancias.plot(kind="barh", color="teal")
plt.xlabel("importancia")
plt.title("Importancia de variables -- XGBoost afinado (top 15)")
plt.tight_layout()
plt.show()

print(importancias.sort_values(ascending=False))

## Parte 10 — Comparación de los modelos

Reúnan, sobre el conjunto de **prueba**, todos los modelos construidos en la actividad.

In [ ]:
modelos = [
    ("Regresión logística", log_reg),
    ("Árbol sin restricciones", arbol_default),
    ("XGBoost (por defecto)", xgb_default),
    ("XGBoost (parada temprana)", xgb_early),
    ("XGBoost (ajustado)", xgb_tuned),
]

filas = []
for nombre, modelo in modelos:
    proba = modelo.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    filas.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test, pred),
        "precision": precision_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "f1": f1_score(y_test, pred),
        "roc_auc": roc_auc_score(y_test, proba),
    })

print(pd.DataFrame(filas).round(4).to_string(index=False))

## Preguntas para discusión

1. En la actividad de **regresión**, XGBoost con hiperparámetros por defecto superaba claramente a la regresión lineal y al árbol. Aquí, con los valores por defecto, **iguala o queda por debajo** de la regresión logística. ¿Qué características de este problema o de este conjunto de datos lo explican?

2. En el ejemplo de la Parte 0, el pseudo-residual no es `y − predicción`, como en regresión, sino `y − σ(predicción)`. Expliquen por qué esta expresión es apropiada cuando `y` solo puede tomar los valores 0 o 1.

3. En la Parte 6, aumentar `scale_pos_weight` apenas modificó el **ROC-AUC**, pero alteró de forma notable la **matriz de confusión**. ¿Cómo se explica que el modelo mantenga el mismo rendimiento según esa métrica y, a la vez, produzca predicciones distintas?

4. Para el equipo de retención de la empresa, ¿qué error tiene mayor costo: señalar como "en riesgo" a un cliente que no iba a darse de baja (falso positivo) o no detectar a uno que sí (falso negativo)? En función de la respuesta, ¿el umbral de decisión debería situarse por encima o por debajo de 0.5?

5. Analicen la importancia de las variables. ¿Resultan razonables las más relevantes para predecir el abandono en telecomunicaciones? ¿Qué variable no incluida en el conjunto de datos cabría esperar que también fuera relevante?

6. La actividad de regresión evaluaba los modelos con MSE y R². Aquí se utilizan accuracy, precision, recall, F1 y ROC-AUC. ¿Por qué la clasificación requiere varias métricas y la regresión, aparentemente, no?